# NS08 Tutorial B - Random Graphs as a Baseline

**Lecture:** NS08 - Network Models

**Reading:** Newman, *Networks*, Section on random graphs

## Learning goals
- Implement the **Gilbert** model $G(n,p)$ from scratch.
- Distinguish $G(n,p)$ from $G(n,m)$ computationally.
- Observe the giant-component **phase transition** around $\langle k \rangle = 1$.
- Compare a matched ER graph to an observed network and identify what the random baseline explains and what it misses.


In [ ]:
from netsci_utils import *
import itertools
import pandas as pd

set_seeds()
%matplotlib inline


def gilbert_graph_from_scratch(n, p, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for u, v in itertools.combinations(range(n), 2):
        if rng.random() < p:
            G.add_edge(u, v)
    return G


def graph_summary(G, name):
    degrees = np.array([degree for _, degree in G.degree()], dtype=float)
    largest_nodes = max(nx.connected_components(G), key=len)
    largest = G.subgraph(largest_nodes).copy()
    return {
        'network': name,
        'n': G.number_of_nodes(),
        'm': G.number_of_edges(),
        '<k>': degrees.mean(),
        'max k': degrees.max(),
        'kappa': heterogeneity_kappa(G),
        'avg clustering': nx.average_clustering(G),
        'largest component fraction': largest_component_fraction(G),
        'avg path length in LCC': nx.average_shortest_path_length(largest),
    }


def giant_component_experiment(n, c_values, trials=20):
    rows = []
    for c in c_values:
        p = c / (n - 1)
        lcc_values = []
        for trial in range(trials):
            G = nx.erdos_renyi_graph(n, p, seed=RANDOM_SEED + trial)
            lcc_values.append(largest_component_fraction(G))
        rows.append({
            '<k>': c,
            'p': p,
            'mean largest-component fraction': np.mean(lcc_values),
        })
    return pd.DataFrame(rows)

---
## 1. Why start from random graphs?

The lecture uses random graphs as an **analytical baseline**.
They answer a precise question:

> What properties would we see if edges appeared without higher-order structure, but only with a fixed global probability?

We use OpenFlights USA as the observed network we want to benchmark.


In [ ]:
openflights = load_openflights_usa()
observed_df = pd.DataFrame([graph_summary(openflights, 'Observed OpenFlights USA')])
print(observed_df.round(3).to_string(index=False))


---
## 2. Gilbert's model $G(n,p)$ from scratch

In Gilbert's model, each possible edge appears independently with probability $p$.
That independence assumption is the core modelling choice.


In [ ]:
G_small = gilbert_graph_from_scratch(n=18, p=0.15, seed=RANDOM_SEED)
pos_small = nx.spring_layout(G_small, seed=RANDOM_SEED)
plot_graph(
    G_small,
    pos=pos_small,
    with_labels=True,
    node_size=500,
    title='One sample from Gilbert\'s random graph model G(n,p)',
)

print_network_stats(G_small)
print(f"Expected number of edges = p * n * (n-1) / 2 = {0.15 * 18 * 17 / 2:.2f}")
print(f"Expected average degree  = p * (n-1)             = {0.15 * 17:.2f}")


---
## 3. The phase transition around $\langle k \rangle = 1$

For $G(n,p)$, the expected average degree is
$$
\langle k \rangle \approx p(n-1).
$$

The lecture says a giant component appears around $\langle k \rangle = 1$. We test that numerically.


In [ ]:
c_values = np.linspace(0.2, 4.0, 15)
transition_df = giant_component_experiment(n=400, c_values=c_values, trials=20)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(
    transition_df['<k>'],
    transition_df['mean largest-component fraction'],
    marker='o',
    linewidth=2,
    color=CATEGORY_PALETTE['blue'],
)
ax.axvline(1.0, color=CATEGORY_PALETTE['red'], linewidth=2, linestyle='--')
style_axis(
    ax,
    title='ER giant-component phase transition',
    xlabel='Expected average degree <k>',
    ylabel='Mean largest-component fraction',
)
plt.show()

print(transition_df.round(3).to_string(index=False))


**Interpretation.**
- Below $\langle k \rangle = 1$, exploration tends to die out.
- Above that threshold, the network can sustain a macroscopic connected component.
- That is the branching-process intuition behind the ER phase transition.


---
## 4. $G(n,p)$ versus $G(n,m)$

Both are random graph families, but they randomize different quantities:
- in $G(n,p)$, the number of edges is random,
- in $G(n,m)$, the number of edges is fixed exactly.


In [ ]:
n_test = 120
mean_degree_target = 6
p_test = mean_degree_target / (n_test - 1)
m_test = round(p_test * n_test * (n_test - 1) / 2)

gnp_edge_counts = []
gnm_edge_counts = []
for seed in range(200):
    gnp_edge_counts.append(nx.gnp_random_graph(n_test, p_test, seed=seed).number_of_edges())
    gnm_edge_counts.append(nx.gnm_random_graph(n_test, m_test, seed=seed).number_of_edges())

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.hist(gnp_edge_counts, bins=18, alpha=0.65, color=CATEGORY_PALETTE['blue'], label='G(n,p)')
ax.hist(gnm_edge_counts, bins=18, alpha=0.55, color=CATEGORY_PALETTE['orange'], label='G(n,m)')
style_axis(
    ax,
    title='Edge-count variability in G(n,p) and G(n,m)',
    xlabel='Number of edges',
    ylabel='Count over 200 trials',
    legend=True,
)
plt.show()

print(f"Target edge count m = {m_test}")
print(f"Mean edges in G(n,p): {np.mean(gnp_edge_counts):.2f}")
print(f"Mean edges in G(n,m): {np.mean(gnm_edge_counts):.2f}")


---
## 5. Match ER to the observed airport network

We now ask the practical null-model question:

> If we keep only the size and mean degree of OpenFlights USA, what does the ER baseline reproduce?


In [ ]:
n_ref = openflights.number_of_nodes()
avg_degree_ref = np.mean([degree for _, degree in openflights.degree()])
p_ref = avg_degree_ref / (n_ref - 1)
er_matched = nx.erdos_renyi_graph(n_ref, p_ref, seed=RANDOM_SEED)

comparison_df = pd.DataFrame([
    graph_summary(openflights, 'Observed OpenFlights USA'),
    graph_summary(er_matched, 'Matched ER baseline'),
])
print(comparison_df.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
plot_ccdf(
    [degree for _, degree in openflights.degree()],
    ax=ax,
    label='OpenFlights USA',
    color=CATEGORY_PALETTE['blue'],
    markersize=3,
)
plot_ccdf(
    [degree for _, degree in er_matched.degree()],
    ax=ax,
    label='Matched ER',
    color=CATEGORY_PALETTE['green'],
    markersize=3,
)
ax.set_title('Degree CCDF: observed network versus matched ER baseline')
ax.legend(frameon=False)
plt.show()


## Takeaway

- Random graphs are the correct **baseline** for understanding what pure chance and fixed global density produce.
- They explain giant-component emergence and short paths surprisingly well.
- They do **not** explain the broad degree heterogeneity or high clustering of the observed airport network.
- This is why NS08 moves next to a model that keeps local neighbourhood structure while introducing shortcuts.
